# Kyrgyzstan Education Statistics — Pure Statistical Analysis

**No visualisations. All results expressed as numerical statistics, tests, and tables.**

Datasets: `literacy.csv`, `students_cis.csv`, `students_non_cis.csv`, `education_budget.csv`

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import linregress, shapiro, mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 30)
print('Libraries loaded.')

Libraries loaded.


In [2]:
literacy_raw  = pd.read_csv('../data/literacy.csv',        encoding='utf-8-sig')
cis_raw       = pd.read_csv('../data/students_cis.csv',     encoding='utf-8-sig')
non_cis_raw   = pd.read_csv('../data/students_non_cis.csv', encoding='utf-8-sig')
budget_raw    = pd.read_csv('../data/education_budget.csv', encoding='utf-8-sig')
print('Files loaded. Shapes:', literacy_raw.shape, cis_raw.shape, non_cis_raw.shape, budget_raw.shape)

Files loaded. Shapes: (13, 7) (12, 18) (13, 18) (8, 28)


## 2. Data Cleaning

In [3]:
# ── Literacy ──────────────────────────────────────────────────────────────────
literacy = literacy_raw[['Items','1989','1999','2009','2022']].copy()
literacy.columns = ['Category','1989','1999','2009','2022']
literacy['Category'] = literacy['Category'].str.strip()
for col in ['1989','1999','2009','2022']:
    literacy[col] = pd.to_numeric(literacy[col], errors='coerce')
literacy_clean = literacy[literacy['Category'].notna() & (literacy['Category']!='') & literacy['1989'].notna()].copy()

# ── International students ────────────────────────────────────────────────────
year_cols = [str(y) for y in range(2010, 2025)]

def extract_by_country(df, group_label):
    en_col = df.columns[2]
    skip = {'total','ncluding:','including:','','nan'}
    rows = []
    for _, row in df.iterrows():
        name = str(row[en_col]).strip()
        if name.lower() in skip or 'Including' in name:
            continue
        vals = pd.to_numeric(pd.Series([row[y] for y in year_cols]), errors='coerce').values
        rows.append({'Country': name, 'Group': group_label, **dict(zip(year_cols, vals))})
    return pd.DataFrame(rows)

def extract_totals(df):
    en_col = df.columns[2]
    mask = df[en_col].astype(str).str.strip().str.lower().isin(['total'])
    row = df[mask].iloc[0]
    return pd.to_numeric(row[year_cols], errors='coerce')

cis_countries     = extract_by_country(cis_raw, 'CIS')
non_cis_countries = extract_by_country(non_cis_raw, 'Non-CIS')
countries_df      = pd.concat([cis_countries, non_cis_countries], ignore_index=True)

cis_total     = extract_totals(cis_raw)
non_cis_total = extract_totals(non_cis_raw)
students_df   = pd.DataFrame({'Year':[int(y) for y in year_cols],
                              'CIS':cis_total.values, 'Non_CIS':non_cis_total.values})
students_df['Total']      = students_df['CIS'] + students_df['Non_CIS']
students_df['CIS_pct']    = students_df['CIS']     / students_df['Total'] * 100
students_df['NonCIS_pct'] = students_df['Non_CIS'] / students_df['Total'] * 100

# ── Budget ────────────────────────────────────────────────────────────────────
budget = budget_raw.copy()
en_col_b = budget.columns[2]
budget[en_col_b] = budget[en_col_b].astype(str).str.strip()
b_years = [str(y) for y in range(2001, 2025)]
for col in b_years:
    if col in budget.columns:
        budget[col] = pd.to_numeric(budget[col], errors='coerce')

def get_budget_row(kw):
    mask = budget[en_col_b].str.lower().str.contains(kw, na=False)
    return budget[mask].iloc[0][b_years].astype(float) if mask.any() else None

budget_ts = pd.DataFrame({'Year':[int(y) for y in b_years],
    'Total_mln_som': get_budget_row('total').values,
    'Preschool':     get_budget_row('pre-school').values if get_budget_row('pre-school') is not None else np.nan,
    'Higher':        get_budget_row('higher').values if get_budget_row('higher') is not None else np.nan,
})
print('Data cleaned. students_df:', students_df.shape, '| budget_ts:', budget_ts.shape)

Data cleaned. students_df: (15, 6) | budget_ts: (24, 4)


## 3. Descriptive Statistics

In [4]:
print('=== 3.1 International Students — Descriptive Statistics ===')
display(students_df[['CIS','Non_CIS','Total']].describe().round(1))

print('\n=== 3.2 Year-on-Year Growth Rates (%) ===')
growth = students_df[['Year','CIS','Non_CIS','Total']].copy()
growth['CIS_growth%']     = growth['CIS'].pct_change() * 100
growth['NonCIS_growth%']  = growth['Non_CIS'].pct_change() * 100
growth['Total_growth%']   = growth['Total'].pct_change() * 100
display(growth.round(2))

print('\n=== 3.3 Students by Country — Total 2010-2024 ===')
country_totals = countries_df.copy()
country_totals['Total_2010_2024'] = country_totals[year_cols].sum(axis=1)
country_totals['Mean_per_year']   = country_totals[year_cols].mean(axis=1).round(1)
country_totals['Max_year_value']  = country_totals[year_cols].max(axis=1)
country_totals['Min_year_value']  = country_totals[year_cols].min(axis=1)
display(country_totals[['Country','Group','Total_2010_2024','Mean_per_year','Max_year_value','Min_year_value']].sort_values('Total_2010_2024', ascending=False).reset_index(drop=True))

=== 3.1 International Students — Descriptive Statistics ===


,CIS,Non_CIS,Total
count,15.00,15.00,15.00
mean,"19,903.90","12,240.90","32,144.90"
std,"16,900.10","9,141.60","25,173.40"
min,"7,068.00","3,099.00","10,167.00"
25%,"8,194.50","3,862.00","12,951.50"
50%,"8,908.00","8,881.00","16,534.00"
75%,"27,563.00","21,563.50","52,690.00"
max,"57,103.00","25,774.00","80,701.00"



=== 3.2 Year-on-Year Growth Rates (%) ===


,Year,CIS,Non_CIS,Total,CIS_growth%,NonCIS_growth%,Total_growth%
0,2010,"9,814.00","3,366.00","13,180.00",NaN,NaN,NaN
1,2011,"7,068.00","3,099.00","10,167.00",-27.98,-7.93,-22.86
2,2012,"7,977.00","3,286.00","11,263.00",12.86,6.03,10.78
3,2013,"8,195.00","3,467.00","11,662.00",2.73,5.51,3.54
4,2014,"8,466.00","4,257.00","12,723.00",3.31,22.79,9.10
5,2015,"8,908.00","5,627.00","14,535.00",5.22,32.18,14.24
6,2016,"8,194.00","6,520.00","14,714.00",-8.02,15.87,1.23
7,2017,"7,653.00","8,881.00","16,534.00",-6.60,36.21,12.37
8,2018,"8,764.00","10,862.00","19,626.00",14.52,22.31,18.70
9,2019,"21,049.00","15,547.00","36,596.00",140.18,43.13,86.47



=== 3.3 Students by Country — Total 2010-2024 ===


,Country,Group,Total_2010_2024,Mean_per_year,Max_year_value,Min_year_value
0,Uzbekistan,CIS,"200,062.00","13,337.50","51,605.00",519.00
1,India,Non-CIS,"110,482.00","7,365.50","15,306.00",581.00
2,Pakistan,Non-CIS,"50,200.00","3,346.70","9,589.00",390.00
3,Kazakstan,CIS,"49,282.00","3,285.50","5,184.00","1,985.00"
4,Russia,CIS,"23,255.00","1,550.30","2,748.00",818.00
5,Tajikistan,CIS,"20,830.00","1,388.70","2,439.00",450.00
6,Turkey,Non-CIS,"9,027.00",601.80,793.00,425.00
7,Other countries,Non-CIS,"5,438.00",362.50,"1,130.00",43.00
8,China,Non-CIS,"5,150.00",343.30,910.00,99.00
9,Turkmenstan,CIS,"3,934.00",262.30,"1,567.00",51.00


In [5]:
print('=== 3.4 Education Budget — Descriptive Statistics (mln som) ===')
display(budget_ts[['Total_mln_som','Preschool','Higher']].describe().round(1))

print('\n=== 3.5 Budget Growth Summary ===')
b_start = budget_ts['Total_mln_som'].iloc[0]
b_end   = budget_ts['Total_mln_som'].iloc[-1]
b_cagr  = (b_end / b_start) ** (1 / (len(budget_ts)-1)) - 1
print(f'  Budget 2001:   {b_start:,.0f} mln som')
print(f'  Budget 2024:   {b_end:,.0f} mln som')
print(f'  Nominal growth: {(b_end/b_start - 1)*100:.1f}%')
print(f'  CAGR (2001-2024): {b_cagr*100:.2f}% per year')

print('\n=== 3.6 Student Enrollment Summary ===')
for grp, col in [('CIS','CIS'),('Non-CIS','Non_CIS'),('Total','Total')]:
    s = students_df[col]
    cagr = (s.iloc[-1]/s.iloc[0])**(1/(len(s)-1))-1
    print(f'  {grp:8s} | 2010: {s.iloc[0]:>7,.0f} | 2024: {s.iloc[-1]:>7,.0f} | Peak: {s.max():>7,.0f} ({students_df.loc[s.idxmax(),"Year"]}) | CAGR: {cagr*100:.2f}%')

=== 3.4 Education Budget — Descriptive Statistics (mln som) ===


,Total_mln_som,Preschool,Higher
count,24.00,24.00,24.00
mean,"28,172.40","3,635.00","3,654.50"
std,"24,598.80","3,978.80","2,922.70"
min,"2,847.60",190.00,490.30
25%,"8,461.30",497.80,"1,378.40"
50%,"23,507.80","2,364.80","2,917.60"
75%,"38,068.80","5,230.90","5,239.70"
max,"83,924.40","14,040.10","11,727.00"



=== 3.5 Budget Growth Summary ===
  Budget 2001:   2,848 mln som
  Budget 2024:   83,924 mln som
  Nominal growth: 2847.2%
  CAGR (2001-2024): 15.85% per year

=== 3.6 Student Enrollment Summary ===
  CIS      | 2010:   9,814 | 2024:  25,073 | Peak:  57,103 (2021) | CAGR: 6.93%
  Non-CIS  | 2010:   3,366 | 2024:  24,480 | Peak:  25,774 (2023) | CAGR: 15.23%
  Total    | 2010:  13,180 | 2024:  49,553 | Peak:  80,701 (2021) | CAGR: 9.92%


## 4. Normality Tests (Shapiro-Wilk)

In [6]:
print('=== 4. Shapiro-Wilk Normality Tests (H0: data is normally distributed) ===')
series_to_test = {
    'CIS students':            students_df['CIS'],
    'Non-CIS students':        students_df['Non_CIS'],
    'Total students':          students_df['Total'],
    'Education budget':        budget_ts['Total_mln_som'],
    'CIS YoY growth%':         students_df['CIS'].pct_change().dropna()*100,
    'Non-CIS YoY growth%':     students_df['Non_CIS'].pct_change().dropna()*100,
}
rows = []
for name, s in series_to_test.items():
    w, p = shapiro(s.dropna())
    rows.append({'Series': name, 'W-statistic': round(w,4), 'p-value': round(p,4),
                 'Normal (α=0.05)': 'Yes' if p > 0.05 else 'No'})
display(pd.DataFrame(rows))

=== 4. Shapiro-Wilk Normality Tests (H0: data is normally distributed) ===


,Series,W-statistic,p-value,Normal (α=0.05)
0,CIS students,0.76,0.00,No
1,Non-CIS students,0.83,0.01,No
2,Total students,0.80,0.00,No
3,Education budget,0.86,0.00,No
4,CIS YoY growth%,0.77,0.00,No
5,Non-CIS YoY growth%,0.97,0.86,Yes


## 5. Correlation Analysis

In [7]:
merged = students_df.merge(budget_ts[budget_ts['Year']>=2010][['Year','Total_mln_som']], on='Year')

print('=== 5.1 Pearson Correlations ===')
pairs = [
    ('Total students', 'Education budget', 'Total', 'Total_mln_som'),
    ('CIS students',   'Education budget', 'CIS',   'Total_mln_som'),
    ('Non-CIS students','Education budget','Non_CIS','Total_mln_som'),
    ('CIS students',   'Non-CIS students', 'CIS',   'Non_CIS'),
]
for lx, ly, cx, cy in pairs:
    r, p = stats.pearsonr(merged[cx], merged[cy])
    print(f'  {lx:25s} vs {ly:25s} | r = {r:+.4f} | p = {p:.4f} | {"Significant" if p<0.05 else "Not significant"} (α=0.05)')

print('\n=== 5.2 Spearman Correlations (rank-based) ===')
for lx, ly, cx, cy in pairs:
    r, p = spearmanr(merged[cx], merged[cy])
    print(f'  {lx:25s} vs {ly:25s} | ρ = {r:+.4f} | p = {p:.4f} | {"Significant" if p<0.05 else "Not significant"} (α=0.05)')

print('\n=== 5.3 Full Correlation Matrix ===')
corr_df = merged[['CIS','Non_CIS','Total','Total_mln_som']].copy()
corr_df.columns = ['CIS Students','Non-CIS Students','Total Students','Education Budget']
display(corr_df.corr().round(4))

=== 5.1 Pearson Correlations ===
  Total students            vs Education budget          | r = +0.7373 | p = 0.0017 | Significant (α=0.05)
  CIS students              vs Education budget          | r = +0.6064 | p = 0.0166 | Significant (α=0.05)
  Non-CIS students          vs Education budget          | r = +0.9093 | p = 0.0000 | Significant (α=0.05)
  CIS students              vs Non-CIS students          | r = +0.8561 | p = 0.0000 | Significant (α=0.05)

=== 5.2 Spearman Correlations (rank-based) ===
  Total students            vs Education budget          | ρ = +0.9036 | p = 0.0000 | Significant (α=0.05)
  CIS students              vs Education budget          | ρ = +0.7250 | p = 0.0022 | Significant (α=0.05)
  Non-CIS students          vs Education budget          | ρ = +0.9786 | p = 0.0000 | Significant (α=0.05)
  CIS students              vs Non-CIS students          | ρ = +0.7893 | p = 0.0005 | Significant (α=0.05)

=== 5.3 Full Correlation Matrix ===


,CIS Students,Non-CIS Students,Total Students,Education Budget
CIS Students,1.00,0.86,0.98,0.61
Non-CIS Students,0.86,1.00,0.94,0.91
Total Students,0.98,0.94,1.00,0.74
Education Budget,0.61,0.91,0.74,1.00


## 6. Linear Regression: Budget → Total Students

In [8]:
from scipy.stats import linregress

X = merged['Total_mln_som'].values
Y = merged['Total'].values

slope, intercept, r_value, p_value, std_err = linregress(X, Y)
n = len(X)
r2 = r_value**2
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - 2)

# Residuals
Y_pred   = slope * X + intercept
residuals = Y - Y_pred
rmse      = np.sqrt(np.mean(residuals**2))
mae       = np.mean(np.abs(residuals))

# 95% CI for slope
t_crit = stats.t.ppf(0.975, df=n-2)
ci_low  = slope - t_crit * std_err
ci_high = slope + t_crit * std_err

print('=== 6. Simple Linear Regression: Total Students ~ Education Budget ===')
print(f'  n               = {n}')
print(f'  Slope           = {slope:.5f}  (students per 1 mln som)')
print(f'  95% CI slope    = [{ci_low:.5f}, {ci_high:.5f}]')
print(f'  Intercept       = {intercept:,.2f}')
print(f'  R               = {r_value:.4f}')
print(f'  R²              = {r2:.4f}')
print(f'  Adjusted R²     = {adj_r2:.4f}')
print(f'  p-value         = {p_value:.6f}')
print(f'  Std Error       = {std_err:.5f}')
print(f'  RMSE            = {rmse:,.0f}')
print(f'  MAE             = {mae:,.0f}')

print('\n=== 6.1 Residuals per Year ===')
resid_df = merged[['Year','Total','Total_mln_som']].copy()
resid_df['Predicted'] = Y_pred.round(0)
resid_df['Residual']  = residuals.round(0)
resid_df['Resid%']    = (residuals / Y * 100).round(2)
display(resid_df)

=== 6. Simple Linear Regression: Total Students ~ Education Budget ===
  n               = 15
  Slope           = 0.82753  (students per 1 mln som)
  95% CI slope    = [0.37320, 1.28185]
  Intercept       = -1,935.92
  R               = 0.7373
  R²              = 0.5436
  Adjusted R²     = 0.5085
  p-value         = 0.001709
  Std Error       = 0.21030
  RMSE            = 16,430
  MAE             = 12,825

=== 6.1 Residuals per Year ===


,Year,Total,Total_mln_som,Predicted,Residual,Resid%
0,2010,"13,180.00","12,822.90","8,675.00","4,505.00",34.18
1,2011,"10,167.00","19,420.40","14,135.00","-3,968.00",-39.03
2,2012,"11,263.00","22,925.90","17,036.00","-5,773.00",-51.26
3,2013,"11,662.00","24,089.70","17,999.00","-6,337.00",-54.34
4,2014,"12,723.00","25,915.40","19,510.00","-6,787.00",-53.34
5,2015,"14,535.00","29,995.00","22,886.00","-8,351.00",-57.45
6,2016,"14,714.00","36,299.30","28,103.00","-13,389.00",-90.99
7,2017,"16,534.00","37,387.80","29,004.00","-12,470.00",-75.42
8,2018,"19,626.00","37,518.60","29,112.00","-9,486.00",-48.33
9,2019,"36,596.00","39,719.50","30,933.00","5,663.00",15.47


## 7. Hypothesis Test — CIS vs Non-CIS Growth Rates

In [9]:
cis_growth_rates     = students_df['CIS'].pct_change().dropna().replace([np.inf,-np.inf], np.nan).dropna() * 100
non_cis_growth_rates = students_df['Non_CIS'].pct_change().dropna().replace([np.inf,-np.inf], np.nan).dropna() * 100

print('=== 7.1 Descriptive Statistics of Annual Growth Rates ===')
for name, s in [('CIS', cis_growth_rates), ('Non-CIS', non_cis_growth_rates)]:
    print(f'  {name:8s} | n={len(s)} | Mean={s.mean():.2f}% | Median={s.median():.2f}% | Std={s.std():.2f}% | Min={s.min():.2f}% | Max={s.max():.2f}%')

print('\n=== 7.2 Welch t-test (unequal variances) — one-tailed ===')
print('  H0: mean growth Non-CIS ≤ mean growth CIS')
print('  H1: mean growth Non-CIS >  mean growth CIS')
t_stat, p_two = stats.ttest_ind(non_cis_growth_rates, cis_growth_rates, equal_var=False)
p_one = p_two / 2
print(f'  t-statistic = {t_stat:.4f}')
print(f'  p (two-tailed) = {p_two:.4f}')
print(f'  p (one-tailed) = {p_one:.4f}')
print(f'  Decision (α=0.05): {"Reject H0 — Non-CIS grew significantly faster" if p_one < 0.05 else "Fail to reject H0"}')

print('\n=== 7.3 Mann-Whitney U test (non-parametric alternative) ===')
u_stat, p_mw = mannwhitneyu(non_cis_growth_rates, cis_growth_rates, alternative='greater')
print(f'  U-statistic = {u_stat:.1f}')
print(f'  p-value     = {p_mw:.4f}')
print(f'  Decision (α=0.05): {"Reject H0" if p_mw < 0.05 else "Fail to reject H0"}')

print('\n=== 7.4 Effect Size (Cohen\'s d) ===')
pooled_std = np.sqrt((cis_growth_rates.std()**2 + non_cis_growth_rates.std()**2) / 2)
cohens_d   = (non_cis_growth_rates.mean() - cis_growth_rates.mean()) / pooled_std
print(f'  Cohen\'s d = {cohens_d:.4f}  ({"small" if abs(cohens_d)<0.5 else "medium" if abs(cohens_d)<0.8 else "large"} effect)')

=== 7.1 Descriptive Statistics of Annual Growth Rates ===
  CIS      | n=14 | Mean=14.59% | Median=3.02% | Std=50.02% | Min=-35.16% | Max=140.18%
  Non-CIS  | n=14 | Mean=16.19% | Median=18.35% | Std=15.47% | Min=-7.93% | Max=43.13%

=== 7.2 Welch t-test (unequal variances) — one-tailed ===
  H0: mean growth Non-CIS ≤ mean growth CIS
  H1: mean growth Non-CIS >  mean growth CIS
  t-statistic = 0.1141
  p (two-tailed) = 0.9107
  p (one-tailed) = 0.4553
  Decision (α=0.05): Fail to reject H0

=== 7.3 Mann-Whitney U test (non-parametric alternative) ===
  U-statistic = 135.0
  p-value     = 0.0468
  Decision (α=0.05): Reject H0

=== 7.4 Effect Size (Cohen's d) ===
  Cohen's d = 0.0431  (small effect)


## 8. Literacy Analysis

In [10]:
print('=== 8.1 Literacy Data — All Categories ===')
display(literacy_clean.set_index('Category'))

print('\n=== 8.2 Percentage Change Between Census Years ===')
for col_a, col_b in [('1989','1999'),('1999','2009'),('2009','2022')]:
    df = literacy_clean.copy()
    df[f'Change {col_a}-{col_b}%'] = ((df[col_b] - df[col_a]) / df[col_a] * 100).round(1)
    print(f'\n  {col_a} → {col_b}:')
    display(df[['Category', col_a, col_b, f'Change {col_a}-{col_b}%']])

print('\n=== 8.3 Key Figures ===')
higher = literacy_clean[literacy_clean['Category'].str.contains('Higher professional', na=False)].iloc[0]
illit  = literacy_clean[literacy_clean['Category'].str.lower().str.strip()=='illiterates'].iloc[0]
print(f'  Higher education 1989→2022: {higher["1989"]:,.0f} → {higher["2022"]:,.0f}  (+{(higher["2022"]/higher["1989"]-1)*100:.1f}%)')
print(f'  Illiterates      1989→2022: {illit["1989"]:,.0f} → {illit["2022"]:,.0f}  ({(illit["2022"]/illit["1989"]-1)*100:+.1f}%)')

=== 8.1 Literacy Data — All Categories ===


,1989,1999,2009,2022
Category,,,,
Total population with education at age 15 years and over,"2,661,901.00","3,090,680.00","3,738,226.00","4,645,716.00"
Higher professional education,"251,246.00","324,414.00","463,346.00","1,134,147.00"
Incomplete higher education,"43,245.00","47,706.00","133,196.00","226,059.00"
Secondary professional education,"418,716.00","333,181.00","264,208.00","819,150.00"
Secondary general education1,"1,040,494.00","1,545,626.00","1,968,846.00","1,607,987.00"
Incomplete secondary education,"489,786.00","566,351.00","443,594.00","442,062.00"
Elementary education,"240,922.00","193,767.00","203,717.00","182,196.00"
Formal education,"177,492.00","79,635.00","63,843.00","49,152.00"
Illiterates,"80,417.00","40,129.00","28,358.00","86,330.00"



=== 8.2 Percentage Change Between Census Years ===

  1989 → 1999:


,Category,1989,1999,Change 1989-1999%
0,Total population with education at age 15 yea...,"2,661,901.00","3,090,680.00",16.10
2,Higher professional education,"251,246.00","324,414.00",29.10
3,Incomplete higher education,"43,245.00","47,706.00",10.30
4,Secondary professional education,"418,716.00","333,181.00",-20.40
6,Secondary general education1,"1,040,494.00","1,545,626.00",48.50
7,Incomplete secondary education,"489,786.00","566,351.00",15.60
8,Elementary education,"240,922.00","193,767.00",-19.60
9,Formal education,"177,492.00","79,635.00",-55.10
11,Illiterates,"80,417.00","40,129.00",-50.10



  1999 → 2009:


,Category,1999,2009,Change 1999-2009%
0,Total population with education at age 15 yea...,"3,090,680.00","3,738,226.00",21.00
2,Higher professional education,"324,414.00","463,346.00",42.80
3,Incomplete higher education,"47,706.00","133,196.00",179.20
4,Secondary professional education,"333,181.00","264,208.00",-20.70
6,Secondary general education1,"1,545,626.00","1,968,846.00",27.40
7,Incomplete secondary education,"566,351.00","443,594.00",-21.70
8,Elementary education,"193,767.00","203,717.00",5.10
9,Formal education,"79,635.00","63,843.00",-19.80
11,Illiterates,"40,129.00","28,358.00",-29.30



  2009 → 2022:


,Category,2009,2022,Change 2009-2022%
0,Total population with education at age 15 yea...,"3,738,226.00","4,645,716.00",24.30
2,Higher professional education,"463,346.00","1,134,147.00",144.80
3,Incomplete higher education,"133,196.00","226,059.00",69.70
4,Secondary professional education,"264,208.00","819,150.00",210.00
6,Secondary general education1,"1,968,846.00","1,607,987.00",-18.30
7,Incomplete secondary education,"443,594.00","442,062.00",-0.30
8,Elementary education,"203,717.00","182,196.00",-10.60
9,Formal education,"63,843.00","49,152.00",-23.00
11,Illiterates,"28,358.00","86,330.00",204.40



=== 8.3 Key Figures ===
  Higher education 1989→2022: 251,246 → 1,134,147  (+351.4%)
  Illiterates      1989→2022: 80,417 → 86,330  (+7.4%)


## 9. Summary Table

In [11]:
summary = pd.DataFrame([
    ['CIS students 2010',               f'{students_df["CIS"].iloc[0]:,.0f}'],
    ['CIS students 2024',               f'{students_df["CIS"].iloc[-1]:,.0f}'],
    ['CIS CAGR 2010-2024',              f'{((students_df["CIS"].iloc[-1]/students_df["CIS"].iloc[0])**(1/14)-1)*100:.2f}%'],
    ['Non-CIS students 2010',           f'{students_df["Non_CIS"].iloc[0]:,.0f}'],
    ['Non-CIS students 2024',           f'{students_df["Non_CIS"].iloc[-1]:,.0f}'],
    ['Non-CIS CAGR 2010-2024',          f'{((students_df["Non_CIS"].iloc[-1]/students_df["Non_CIS"].iloc[0])**(1/14)-1)*100:.2f}%'],
    ['Peak total enrollment (year)',    f'{students_df["Total"].max():,.0f} ({students_df.loc[students_df["Total"].idxmax(),"Year"]})'],
    ['Pearson r (Total vs Budget)',     f'{stats.pearsonr(merged["Total"], merged["Total_mln_som"])[0]:.4f}'],
    ['Regression R²',                   f'{linregress(merged["Total_mln_som"], merged["Total"]).rvalue**2:.4f}'],
    ['Welch t-test p (one-tailed)',     f'{stats.ttest_ind(non_cis_growth_rates, cis_growth_rates, equal_var=False)[1]/2:.4f}'],
    ['Budget 2001 (mln som)',           f'{budget_ts["Total_mln_som"].iloc[0]:,.0f}'],
    ['Budget 2024 (mln som)',           f'{budget_ts["Total_mln_som"].iloc[-1]:,.0f}'],
    ['Budget CAGR 2001-2024',           f'{((budget_ts["Total_mln_som"].iloc[-1]/budget_ts["Total_mln_som"].iloc[0])**(1/23)-1)*100:.2f}%'],
], columns=['Metric','Value'])
display(summary)

,Metric,Value
0,CIS students 2010,"9,814"
1,CIS students 2024,"25,073"
2,CIS CAGR 2010-2024,6.93%
3,Non-CIS students 2010,"3,366"
4,Non-CIS students 2024,"24,480"
5,Non-CIS CAGR 2010-2024,15.23%
6,Peak total enrollment (year),"80,701 (2021)"
7,Pearson r (Total vs Budget),0.7373
8,Regression R²,0.5436
9,Welch t-test p (one-tailed),0.4553
